## Calculates daily OHLCV and metrics
- from silver `workspace.market_silver.intraday_candles` to `workspace.market_gold.daily_prices`
- aggregates: first open, max high, min low, last close, sums volume
- metrics: prev_close, daily_return, moving average of 7 days
- reuses transformations from `market_tracker.spark.transformations`
- calculated separately for minute and hour interval
- only regular sessions (ext_hours = false)
- overwrites gold each time, because current metrics depend on previous days


In [0]:
import os
import sys

sys.path.append(os.path.abspath('../../src'))

from market_tracker.spark.transformations import aggregate_daily, add_daily_metrics
from pyspark.sql import functions as F

In [0]:
candles = spark.read.table('workspace.market_silver.intraday_candles')
candles_minute = candles.filter(F.col('interval') == 'minute')
candles_hour = candles.filter(F.col('interval') == 'hour')


In [0]:
aggregated_minute = aggregate_daily(candles_minute.withColumnRenamed("candle_ts", 'timestamp'))
aggregated_hour = aggregate_daily(candles_hour.withColumnRenamed("candle_ts", 'timestamp'))

In [0]:
metrics_minute = add_daily_metrics(aggregated_minute).withColumn('source_interval', F.lit('minute'))
metrics_hour = add_daily_metrics(aggregated_hour).withColumn('source_interval', F.lit('hour'))


In [0]:
metrics = metrics_minute.unionByName(metrics_hour)

In [0]:
metrics.write.saveAsTable('workspace.market_gold.daily_prices', mode='Overwrite')
spark.sql("COMMENT ON TABLE workspace.market_gold.daily_prices IS 'silver data one row per (ticker, source_interval, date), aggregated daily with computed metrics: daily_return and moving average of 7 days. regular trading hours only (ext_hours = false)'")

## Checks

In [0]:
# interval count sanity check
gold = spark.read.table('workspace.market_gold.daily_prices')
gold.groupBy(F.col('source_interval')).count().display()

In [0]:
# check for duplicates
gold.groupBy(F.col('ticker'), F.col('source_interval'), F.col('date')).count().where(F.col('count') > 1).display()